# CLM-001-R2 — Run All
Automatic preflight → paired 500-step development → isolated analysis → GitHub return.
Successful checks continue automatically; failed checks stop before training. No second launch or mode change.
Use the original bingjiangli/minicells Notebook, T4 x2, Internet, HF_TOKEN and GITHUB_TOKEN. Save Version retains checkpoints.
Exploratory development only; independent confirmation remains sealed.


In [ ]:
from pathlib import Path

SOURCE_REF = "codex/clm001-r2"
ROOT = Path("/kaggle/working/clm001-r2-code")
OUTPUT = Path("/kaggle/working/clm001-r2-output")
ASSETS = Path("/kaggle/working/clm001-r2-assets")

In [ ]:
import importlib.util
import subprocess
import sys


def run(args, **kwargs):
    print("[CLM-001-R2]", " ".join(map(str, args)), flush=True)
    return subprocess.run(list(map(str, args)), check=True, **kwargs)


# Keep Kaggle's installed stack. Install only genuinely missing experiment modules.

requirements = {
    "pytest": "pytest",
    "numpy": "numpy",
    "tokenizers": "tokenizers",
    "huggingface_hub": "huggingface_hub",
    "datasets": "datasets>=2.18,<5",
}
missing = [
    requirement
    for module, requirement in requirements.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    run([sys.executable, "-m", "pip", "install", "--quiet", *missing])
if not (ROOT / ".git").exists():
    ROOT.mkdir(parents=True, exist_ok=True)
    run(["git", "init", "--quiet", ROOT])
    run(["git", "remote", "add", "origin", "https://github.com/ArcheLabs/mini-cells.git"], cwd=ROOT)
run(["git", "sparse-checkout", "init", "--no-cone"], cwd=ROOT)
run(
    [
        "git",
        "sparse-checkout",
        "set",
        "--no-cone",
        "/pyproject.toml",
        "/.github/workflows/clm001-r2-cpu.yml",
        "/src/",
        "/shared/python/",
        "/tests/research/clm001_r2/",
        "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/",
        "/research/projects/cellular-language-models/experiments/006-path-first-clm/",
        "/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/",
        "/research/projects/continual-learning/context/04-continual-learning/scripts/research/",
        "/research/projects/cellular-language-models/context/08-hybrid-clm/scripts/research/",
    ],
    cwd=ROOT,
)
run(["git", "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_REF], cwd=ROOT)
run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=ROOT)
source = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
BASE = ROOT / "research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r2"
OUTPUT = OUTPUT / source[:12]
OUTPUT.mkdir(parents=True, exist_ok=True)
print(
    {
        "experiment": "CLM-001-R2",
        "mode": "auto",
        "source": source,
        "scientific_decision": False,
    },
    flush=True,
)
import os  # noqa: E402

if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient

        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN").strip()
    except Exception as error:
        print(
            "HF_TOKEN unavailable:",
            type(error).__name__,
            "; public dataset access only",
            flush=True,
        )
run(
    [sys.executable, BASE / "run.py", "--mode", "auto", "--output", OUTPUT, "--assets", ASSETS],
    cwd=ROOT,
)
print("Retain with Save Version:", OUTPUT, flush=True)